# Pelajaran 03 - Corak Reka Bentuk Ejen

Dalam pelajaran ini, kita meneroka tiga corak reka bentuk asas untuk membina ejen AI yang berkesan:

1. **Arahan Ejen yang Jelas** — Membentuk prompt yang tepat dan mendefinisikan peranan yang membimbing tingkah laku ejen
2. **Output Berstruktur dengan Model Pydantic** — Memastikan ejen memberikan data yang boleh diramalkan dan disahkan
3. **Ejen Tanggungjawab Tunggal** — Mereka bentuk ejen fokus yang masing-masing melakukan satu perkara dengan baik

Kita akan menggunakan setiap corak pada senario **pencadangan destinasi pelancongan**, secara berperingkat membina sistem yang boleh mencadangkan destinasi, memeriksa ketersediaan, dan mengendalikan logistik.


## Persediaan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Corak 1: Arahan Ejen Yang Jelas

Corak yang paling berkesan juga yang paling mudah: menulis arahan yang jelas dan terperinci untuk ejen anda.

Arahan yang baik mendefinisikan:
- **Siapa** ejen itu (persona dan nada)
- **Apa** yang harus dilakukan (tanggungjawab langkah demi langkah)
- **Bagaimana** ia harus berkelakuan (had dan gaya)

Di bawah, kami mencipta ejen konsjer percutian dengan arahan yang jelas yang membentuk setiap respons yang dihasilkannya.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Corak 2: Output Berstruktur dengan Model Pydantic

Teks bebas berguna untuk perbualan, tetapi sistem hiliran memerlukan data berstruktur.
Dengan memadankan **model Pydantic** dengan **fungsi alat**, kita boleh:

- Menentukan skema tepat untuk output ejen
- Mengesahkan respons secara automatik
- Mengintegrasikan keputusan ejen ke dalam logik aplikasi dengan boleh dipercayai

Kunci untuk penguatkuasaan adalah dengan melepasi `response_format` apabila kita menjalankan ejen. Ini memaksa
model mengembalikan objek `TravelRecommendations` yang disahkan (tersedia pada `response.value`)
dan bukannya teks bebas. Alat `get_destination_details` juga mengembalikan
`DestinationRecommendation` berjenis, jadi data kekal berstruktur dari awal hingga akhir.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Corak 3: Ejen Tanggungjawab Tunggal

Tugasan yang kompleks mendapat manfaat daripada membahagikan kerja kepada beberapa ejen yang fokus, setiap satu dengan tanggungjawab tunggal:

- Seorang **Pakar Destinasi** yang mengetahui tentang tempat dan ketersediaan
- Seorang **Perancang Logistik** yang menguruskan penerbangan, hotel, dan jadual perjalanan

Ini mencerminkan prinsip kejuruteraan perisian *pemisahan kebimbangan* — setiap ejen lebih mudah untuk diuji, diselenggara, dan diperbaiki secara bebas.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Ringkasan

Dalam pelajaran ini kami menerapkan tiga corak reka bentuk agenik kepada senario pencadangan perjalanan:

| Corak | Idea Utama | Manfaat |
|---|---|---|
| **Arahan Jelas** | Takrifkan persona, tanggungjawab, dan kekangan di awal | Tingkah laku agen yang konsisten dan berjenama |
| **Output Berstruktur** | Gunakan model Pydantic sebagai format respons | Keputusan yang disahkan dan boleh dibaca mesin |
| **Tanggungjawab Tunggal** | Berikan setiap agen satu tugas fokus | Lebih mudah untuk menguji, menyelenggara, dan menggabungkan |

Corak-corak ini bergabung secara semula jadi — anda boleh menggabungkan arahan jelas dengan output berstruktur dalam agen bertanggungjawab tunggal untuk membina sistem yang kukuh dan sedia produksi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
